# Video highlight candidate detection

**Recipe 92 · Level 3 (Intermediate) · Decision type: multiframe `Score`**

Sources:

- S02: [TypeSafe AI: Primitives (Questions)](https://docs.typesafe.ai/primitives)
- S03: [TypeSafe AI: Confidence](https://docs.typesafe.ai/confidence)
- S09: [TypeSafe AI: Images](https://docs.typesafe.ai/images)
- S11: [TypeSafe AI: Models and long context](https://docs.typesafe.ai/models)
- S12: [TypeSafe AI: Jev 1.14 release](https://typesafe.ai/blog/jev-1-14)

## What you will build

Build a shortlist of timestamped highlight candidates from three ordered still frames per window. Three independent `Score` questions judge visible raised-arm posture, persistent setting replacement, and a game victory overlay. Python samples frames, combines scores, limits requests, and ranks the proposed windows. A candidate is a suggestion for an editor, not a conclusion that a person is happy or that footage deserves publication.

These are original synthetic illustrations: 26 windows, each with five extracted PNG frames, not photographs or decoded video. Twelve windows select the policy, twelve test it after freezing, and two are unscored demonstrations. All stored probabilities are hand-written, including confident mistakes. The committed run is a pipeline check; it is **not measured live**. Experimental image support uses a pinned `jev-1.14.0` live request, with ordered stills rather than a native video payload (S09, S11, S12).

## Setup and run mode

The backend changes between offline replay and explicit live exploration; the questions and image assets stay the same. A [replay key](../../docs/glossary.md#replay-key) identifies the entire state, questions and referenced image bytes. `WindowScorer` caches by that key so displaying an answer again spends no extra request. The local cap is 26 unique requests; the live backend separately caps HTTP attempts, including retries. See the README for the live budget and recording procedure.

In [ ]:
import json

from IPython.display import display

from jev_cookbook import get_backend, load_helpers
from jev_cookbook.evaluation import (
    accuracy,
    confusion_matrix,
    evaluate_outcomes,
    select_confidence_threshold,
    select_threshold,
)
from jev_cookbook.fixtures import load_inputs, load_labels, responses_path
from jev_cookbook.style import (
    apply_style,
    plot_answer_probabilities,
    plot_confusion_matrix,
    run_header,
    show_answer,
)
from jev_cookbook.vision import run_record

apply_style()
helpers = load_helpers()
examples = load_inputs()
labels = load_labels()
backend = get_backend(fixtures=responses_path(), images=True, model="jev-1.14.0")
scorer = helpers.WindowScorer(backend, max_requests=26)
offline = backend.mode in {"synthetic", "scripted"}
check = " — pipeline check, not a Jev result" if offline else ""
selection = " — validation selection step"
run_header(92, "Video highlight candidate detection", backend=backend, n_examples=len(labels))
print("26 windows; 130 source frames; 3 sampled frames and 3 independent scores per request")

## The state

A window is a short time range in a synthetic program timeline. Python selects the actual source frames nearest three evenly spaced targets, preserving their timestamps: five frames at offsets 0.00, 0.45, 0.95, 1.55 and 2.05 seconds become samples at 0.00, 0.95 and 2.05 seconds. Equal-distance ties go to the earlier source frame. Invalid times or sampling targets that collapse to the same frame are rejected before a request.

Only `frames`, each with a time and an opaque image reference, reaches inference. Local paths, scenario names, split names and annotations stay outside the state. The full five-frame manifest remains available to an editor. For real video, a decoder must first emit upright PNG/JPEG frames and measured timestamps; this lesson starts at that boundary. Events between samples can be missed.

In [ ]:
demo = next(e for e in examples if e.id == "d01")
state, questions, assets = helpers.request(demo.fields)
print(json.dumps(state, indent=2))
print("Sampled window bounds:", helpers.window_interval(demo.fields))
display(helpers.contact_sheet(demo.fields))

## The questions

Each question has four ordered levels, fully defined below. `reaction` asks only about visible posture, not an inferred emotion. `scene_change` separates a persistent setting replacement from camera cuts within a setting and exposure flicker. `victory` uses an explicit trophy-plus-VICTORY visual signal; a SHOP advert or PAUSED overlay does not satisfy it. These independent judgments share one request (S02).

A `Score` is the probability-weighted position on levels 0–3. Its [confidence](../../docs/glossary.md#confidence) summarizes the distribution; it is not a guarantee that an answer is correct (S03). Level 2 names transient or unclear evidence. Python gives that range an explicit uncertain outcome when no stronger signal qualifies.

In [ ]:
for name, question in questions.items():
    print(name, "—", question.instructions)
    for level, criterion in enumerate(question.criteria):
        print(f"  {level}: {criterion}")

## One answer up close

Read the pixels first, then the complete typed reaction answer. Both hands rise above the shoulders and stay there in the final two samples. The face is deliberately neutral; nothing here establishes a mental state. The score, distribution, level legend, confidence and synthetic provenance are all visible before any aggregate.

In [ ]:
demo_result = scorer.decide(demo.fields)
show_answer(demo_result["reaction"])
print(
    "Other independent scores:",
    {name: demo_result[name].score for name in ("scene_change", "victory")},
)
plot_answer_probabilities(demo_result["reaction"], title="Visible raised-arm posture")

A brief flash changes many pixels. It does not establish a persistent new setting. The second demonstration deliberately assigns the scene-change score near level 2, which Python will send to a person rather than interpret as a completed highlight judgment.

In [ ]:
flash_demo = next(e for e in examples if e.id == "d02")
flash_result = scorer.decide(flash_demo.fields)
display(helpers.contact_sheet(flash_demo.fields))
show_answer(flash_result["scene_change"])

## Python's part

The [business threshold](../../docs/glossary.md#business-threshold), the score needed to propose a candidate, is fixed at 2.4 on the 0–3 scale. It encodes this lesson's persistent-signal requirement; it was defined before validation. The largest of the three scores controls candidacy. The [confidence gate](../../docs/glossary.md#confidence-gate) accepts only when the minimum confidence across all three judgments reaches its validation-selected threshold. Taking the minimum is a conservative recipe policy, not a calibrated joint probability.

The [gold label](../../docs/glossary.md#gold-label) is a separately authored visible-event annotation, including the event's time interval and a readable explanation. Validation chooses the lowest observed confidence with perfect raw event accuracy on that small validation pool; those observed confidences are the candidate family. If no candidate satisfies the objective, selection fails explicitly. The rule also sends strength in [1.5, 2.4) to [review](../../docs/glossary.md#review), a deferred answer that an editor must resolve. A resolved low score means `none_suitable`; it is still confidence-gated.

The deterministic comparison uses maximum adjacent-frame mean absolute RGB difference, normalized to [0,1], on exactly the same three samples. Its threshold is selected from observed validation differences by [F1](../../docs/glossary.md#f1), the harmonic mean of positive precision and recall; ties choose the higher threshold. A large exposure change or game pause can fool this baseline, while a small but meaningful posture change may be missed. Pixel difference is a scalar activity measure, not a probability.

Both selections happen below using validation only and are frozen before test requests. Python also derives start/end times, sorts candidates by score then start time then identifier, and caps the simulated editorial shortlist at three. Budget exhaustion returns review before another semantic request can be made. No video is exported.

In [ ]:
validation = [e for e in examples if e.split == "validation"]
val_results = [scorer.decide(e.fields) for e in validation]
val_gold = [labels[e.id]["event"] for e in validation]
val_correct = [helpers.raw_event(r) == g for r, g in zip(val_results, val_gold, strict=True)]
threshold = select_confidence_threshold(
    val_correct, [helpers.confidence(r) for r in val_results], target_accuracy=1.0
)
val_difference = [helpers.pixel_change(e.fields) for e in validation]
baseline_threshold = select_threshold(
    [g == "candidate" for g in val_gold], val_difference, objective="f1"
)
print(f"N={len(validation)}; frozen confidence threshold={threshold:.4f}{selection}{check}")
print(
    f"N={len(validation)}; frozen pixel-difference threshold={baseline_threshold:.6f}{selection}{check}"
)
print("Demo candidate:", helpers.route(demo.id, demo.fields, demo_result, threshold))
print("Demo flash:", helpers.route(flash_demo.id, flash_demo.fields, flash_result, threshold))

## Evaluation

[Coverage](../../docs/glossary.md#coverage) is the fraction of windows resolved; abstention is its complement. [Accuracy](../../docs/glossary.md#accuracy) is the fraction of compared event labels that agree, and [risk](../../docs/glossary.md#risk) is the fraction of resolved labels that are wrong. We print raw event accuracy on all N windows, resolved event accuracy, abstention, and the count of correct resolved windows over N. `evaluate_outcomes` receives actual route acceptance flags, including the temporal ambiguity branch; confidence alone cannot reproduce those decisions.

The baseline resolves every window. Both methods see the same test frames and labels. Event correctness here means candidate versus none suitable for the whole window. The annotated event spans audit the drawings; proposed windows intentionally include the lead-in frame. We do not claim exact event onset detection or infer events outside the sampled evidence. A shortlist cap changes how many candidate suggestions are displayed, not the event-classification denominator.

In [ ]:
def report(rows, results, gold):
    outcomes = [
        helpers.route(e.id, e.fields, r, threshold) for e, r in zip(rows, results, strict=True)
    ]
    raw = [helpers.raw_event(r) for r in results]
    accepted = [o.outcome != helpers.REVIEW for o in outcomes]
    correct = [o.outcome == g for o, g in zip(outcomes, gold, strict=True)]
    metrics = evaluate_outcomes(accepted, correct)
    baseline = [
        "candidate" if helpers.pixel_change(e.fields) >= baseline_threshold else "none_suitable"
        for e in rows
    ]
    return outcomes, metrics, accuracy(gold, raw), accuracy(gold, baseline), baseline


val_outcomes, val_metrics, val_raw, val_baseline, _ = report(validation, val_results, val_gold)
print(
    f"N={len(validation)}; raw event accuracy={val_raw:.4f}; baseline event accuracy={val_baseline:.4f}{selection}{check}"
)
print(
    f"N={len(validation)}; resolved={val_metrics.n_answered}; coverage={val_metrics.coverage:.4f}; abstention={1 - val_metrics.coverage:.4f}; resolved event accuracy={val_metrics.accuracy:.4f}{selection}{check}"
)

In [ ]:
test = [e for e in examples if e.split == "test"]
test_results = [scorer.decide(e.fields) for e in test]
test_gold = [labels[e.id]["event"] for e in test]
test_outcomes = [
    helpers.route(e.id, e.fields, r, threshold) for e, r in zip(test, test_results, strict=True)
]
if backend.mode == "live":
    print("Unrecorded live exploration: held-out metrics withheld; record the requests first.")
else:
    test_outcomes, metrics, raw_accuracy, baseline_accuracy, baseline = report(
        test, test_results, test_gold
    )
    print(f"Test N={len(test)}; raw event accuracy={raw_accuracy:.4f}{check}")
    print(
        f"Test N={len(test)}; resolved={metrics.n_answered}; review={len(test) - metrics.n_answered}; coverage={metrics.coverage:.4f}; abstention={1 - metrics.coverage:.4f}{check}"
    )
    print(
        f"Test N={len(test)}; resolved event accuracy={metrics.accuracy:.4f}; risk={metrics.risk:.4f}{check}"
    )
    print(
        f"Test N={len(test)}; correct resolved windows={sum(o.outcome == g for o, g in zip(test_outcomes, test_gold, strict=True))}/{len(test)}{check}"
    )
    print(
        f"Baseline test N={len(test)}; event accuracy={baseline_accuracy:.4f}; abstention=0.0000{check}"
    )
    false_positives = [
        e.id
        for e, o, g in zip(test, test_outcomes, test_gold, strict=True)
        if o.outcome == "candidate" and g != "candidate"
    ]
    missed = [
        e.id
        for e, o, g in zip(test, test_outcomes, test_gold, strict=True)
        if o.outcome == "none_suitable" and g == "candidate"
    ]
    unresolved_events = [
        e.id
        for e, o, g in zip(test, test_outcomes, test_gold, strict=True)
        if o.outcome == "review" and g == "candidate"
    ]
    print(
        f"Test N={len(test)}; false candidates={false_positives}; missed events={missed}; events in review={unresolved_events}{check}"
    )

In [ ]:
if backend.mode != "live":
    matrix = confusion_matrix(
        test_gold,
        [o.outcome for o in test_outcomes],
        labels=["candidate", "none_suitable", "review"],
    )
    print(f"Test N={len(test)}; gold rows / routed columns: {matrix.labels}{check}")
    print(matrix.matrix)
    display(plot_confusion_matrix(matrix, title=f"Test N={len(test)}{check}"))

### A confident error survives

The fixture `t10` contains only one raised arm. Its stored reaction distribution nonetheless strongly favors level 3, so the frozen policy accepts a false candidate. The same policy accepts `t11` as none suitable even though both arms rise. These are deliberately authored failures, not observed Jev behavior. They show that confidence and sampling rules cannot certify correctness.

In [ ]:
hard = next(e for e in test if e.id == "t10")
hard_result = scorer.decide(hard.fields)
display(helpers.contact_sheet(hard.fields))
show_answer(hard_result["reaction"])
print("Annotation:", labels[hard.id])
print("Actual route:", helpers.route(hard.id, hard.fields, hard_result, threshold))

In [ ]:
selected, review, actions = helpers.shortlist(test_outcomes, limit=3)
print("Ranked proposed windows:", [(o.window, o.start, o.end, o.strength) for o in selected])
print("Simulated action log:")
print(json.dumps(actions.to_dicts(), indent=2))
print("Pending review queue:")
print(json.dumps(review.to_dicts(), indent=2))
print("No media was exported or published. Unique semantic requests:", scorer.requests_made)

## What was and was not measured

The cell below states mode, N, and the returned model and capture date when recorded or live. The committed sample has 12 validation windows, 12 test windows and 2 unscored demonstrations. Its metrics check the pipeline on original synthetic illustrations and invented distributions. They establish no Jev quality, general video performance, latency or cost claim.

`run_record` retains actual API usage only for real recorded/live responses. Missing tokens, inference latency and billed cost remain `None`; replay wall time is never inference latency. A live notebook is unrecorded exploration with no reportable held-out result. Capture `(state, questions, assets)` triples using the README's recorder path, preserve returned model/date/usage, and run a recorded replay before reporting that limited test sample.

In [ ]:
if offline:
    print(f"Provenance: {backend.mode}. Not measured live: a pipeline check, not a Jev result.")
    print(f"N: {len(validation)} validation, {len(test)} test, 2 unscored demos{check}")
else:
    print(f"Mode: {backend.mode}; returned model: {backend.model}")
    print(f"N: {len(validation)} validation, {len(test)} test, 2 unscored demos")
    print("Capture date(s):", getattr(backend, "recorded_dates", ()) or "this unrecorded run")
print("Accounting for one request:", run_record(demo_result, mode=backend.mode))
print("Actual semantic calls:", scorer.requests_made, "; local cap:", scorer.max_requests)

## Next steps

Keep an untouched real test set before changing frame spacing, highlight definitions or confidence policy. Add independent human annotations and realistic camera motion, transitions, long occlusion and short missed events. A decoder integration should preserve presentation timestamps, enforce image limits, and measure actual inference rather than replay. The current small synthetic set cannot select a production editing policy.

Compare [response clarity scoring](../03-response-clarity-scoring/) for a single bounded Score and [incident priority composition](../24-incident-priority-composition/) for independent judgments combined in Python. Follow the [offline/live contract](../../docs/offline-and-live.md) when recording.